# Baseline Benchmark Reproduction

This notebook reproduces the baseline deserialization benchmark for the python-rapidjson repository using the canada.json workload dataset.

The benchmark measures repeated JSON deserialization performance using rapidjson.loads() on a Google Colab CPU runtime.

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

In [ ]:
!git clone https://github.com/python-rapidjson/python-rapidjson.git

In [ ]:
%cd python-rapidjson

In [ ]:
# replace later with final baseline SHA
!git checkout <342ae326780f55f0f71ccb6c2ed3de0c9b6b62a7>

In [ ]:
!git submodule update --init --recursive
!pip install .
!pip install -r requirements-test.txt
!pip install pyinstrument pytest-benchmark

In [ ]:
!pytest tests

In [ ]:
benchmark_code = r'''
import time
import statistics
from pathlib import Path
import rapidjson

JSON_PATH = Path("benchmarks/json/canada.json")

data = JSON_PATH.read_text(encoding="utf-8")

WARMUPS = 3
RUNS = 7
ITERATIONS = 250

def workload():
    for _ in range(ITERATIONS):
        rapidjson.loads(data)

for _ in range(WARMUPS):
    workload()

times = []

for i in range(RUNS):
    start = time.perf_counter()
    workload()
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    print(f"Run {i+1}: {elapsed:.4f}s")

median = statistics.median(times)
iqr = statistics.quantiles(times, n=4)[2] - statistics.quantiles(times, n=4)[0]

print("\\nRESULTS")
print(f"Median: {median:.4f}s")
print(f"IQR: {iqr:.4f}s")
'''

with open("benchmark_canada.py", "w") as f:
    f.write(benchmark_code)

In [ ]:
!python benchmark_canada.py

In [ ]:
!pyinstrument benchmark_canada.py